In [2]:
import os 
from dotenv import load_dotenv
from agents import Agent, AsyncOpenAI, trace, Runner, function_tool, handoff
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from sendgrid.helpers.mail import Email, Mail, Content, To
import sendgrid
import asyncio
from pydantic import BaseModel
from typing import Dict

load_dotenv(override=True)

True

In [3]:
api_key = os.getenv("OPENROUTER_API_KEY")
client = AsyncOpenAI(
    api_key=api_key, 
    base_url="https://openrouter.ai/api/v1",
    default_headers={           # ← this is missing
        "HTTP-Referer": "http://localhost",
        "X-Title": "RecruitAI"
    }
)

model_wrapper = OpenAIChatCompletionsModel(
    model="openai/gpt-4o-mini", 
    openai_client=client
)

print(api_key)


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


In [4]:
class EmailOutput(BaseModel):
    subject: str
    body: str

In [5]:
formal_agent = Agent(
  name="Formal Agent",
  model=model_wrapper,
  instructions="""
    You are a professional recruitment consultant.
    
    Given a candidate's profile and a job role, you must:
    1. Assess whether the candidate is a good fit for the role
    2. Write a formal, professional cold email inviting them to interview
    
    The email should be concise, respectful, and clearly state the next step.
    """,
  output_type=EmailOutput
)

persuasive_agent = Agent(
  name="Persuasive Agent",
  model=model_wrapper,
  instructions="""
    You are a persuasive recruitment consultant.
    
    Given a candidate's profile and a job role, you must:
    1. Assess whether the candidate is a good fit for the role
    2. Write a persuasive email to the candidate, highlighting the benefits of the role and the company
    3. Ask the candidate to apply for the role
    The email should be concise, respectful, and clearly state the next step.
    """,
  output_type=EmailOutput
)

friendly_agent = Agent(
  name="Friendly Agent",
  model=model_wrapper,
  instructions="""
    You are a friendly recruitment consultant.
    
    Given a candidate's profile and a job role, you must:
    1. Be friendly to the candidate
    2. Ask how the candidate is doing 
    3. Ask if they are interested in the role
    4. Ask if they have any questions
    The email should be concise, respectful, and clearly state the next step.
    """,
  output_type=EmailOutput
)

html_cover_letter_agent = Agent(
  name="HTML Cover Letter Agent",
  model=model_wrapper,
  instructions="""
    You are a HTML cover letter writer.
    You convert Emails to HTML.
  """
)

In [ ]:
with trace("Recruiting"):
  instruction = """
    Candidate: John Doe
    Skills: Python, FastAPI, 3 years experience
    Applying for: Senior Software Engineer at TechCorp
  """
  results =  await asyncio.gather(
    Runner.run(formal_agent, instruction),
    Runner.run(persuasive_agent, instruction),
    Runner.run(friendly_agent, instruction)
  )
  outputs = [result.final_output for result in results]

  for i, output in enumerate(outputs):
    print(f"\n{'='*50}")
    print(f"EMAIL {i+1}")
    print(f"{'='*50}\n")
    print(output)

In [6]:
formal_tool = formal_agent.as_tool(
  tool_name="formal_email_writer",
  tool_description="Writes a formal cold email for a candidate"

)
friendly_tool = friendly_agent.as_tool(
  tool_name="friendly_email_writer",
  tool_description="Writes a friendly email for a candidate"
)
persuasive_tool = persuasive_agent.as_tool(
  tool_name="persuasive_email_writer",
  tool_description="Writes a persuasive email for a candidate"
)

html_cover_letter_tool = html_cover_letter_agent.as_tool(
  tool_name="html_cover_letter_writer",
  tool_description="Converts an email to HTML"
)


In [7]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    print("Send email details",subject, html_body)
    """Send an email with the given subject and HTML body"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("galdunxagency@gmail.com")  # put your verified sender here
    to_email = To("ibrahimsheriff999@gmail.com")  # put your recipient here
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print("Email response", response.status_code)
    return "success"


In [ ]:
instruction = """
    Candidate: John Doe
    Skills: Python, FastAPI, 3 years experience
    Applying for: Senior Software Engineer at TechCorp
  """
  
email_agent = Agent(
  name="Email Agent",
  model=model_wrapper,
  instructions="""
    You are an email agent.
    Your convert emails to html and send html emails only
  """,
  tools=[send_email, html_cover_letter_tool],
  handoff_description="Convert an email to HTML and send it",
)

senior_recruiter = Agent(
  name="Senior Recruiter",
  model=model_wrapper,
  instructions="""
    You are a senior recruiter.
    Given a candidate profile:
    1. Call all three email writing tools
    2. Compare the three emails
    3. Pick the best one and explain why
    4. Hand off the winning email to the email agent for sending
    """,
  tools=[formal_tool, friendly_tool, persuasive_tool],
  handoffs=[email_agent]
)


with trace("Senior Recruiter"):
  result = await Runner.run(senior_recruiter, instruction)
  message = result.final_output
  print(message)

SyntaxError: unmatched ')' (3113246866.py, line 16)

In [ ]:
import gradio as gr
import nest_asyncio
nest_asyncio.apply()

async def chat(message, history):
    with trace("Recruiting"):
        result = await Runner.run(senior_recruiter, message)
        return result.final_output

demo = gr.ChatInterface(
    fn=chat,
    title="RecruitAI 🤝",
    description="Tell me who you want to recruit and I'll handle the rest!",
    examples=[
        "Recruit John Doe, Python/FastAPI developer, 3 years experience, for Senior Software Engineer at TechCorp",
        "Recruit Jane Smith, React/Node.js developer, 5 years experience, for Lead Frontend Engineer at StartupXYZ"
    ]
)

demo.launch()